# Primordial 8 TeV Full Compare (Pert vs NPWLC, all forms)
Notebook-style production for `R_{pA}` vs y / pT / centrality in three rapidities, across forms: `new`, `old`, `radius`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

project = Path.cwd()
if not (project / 'primordial_code').exists():
    project = project.parent

sys.path.insert(0, str(project))
sys.path.insert(0, str(project / 'primordial_code'))
sys.path.insert(0, str(project / 'primordial_notebooks'))

import primordial_only_eloss_glauber_test as prim
from primordial_module import ReaderConfig, Style, make_bins_from_width


In [2]:
ENERGY = '8.16'
FORMS = ('new', 'old', 'radius')
MODELS = ('Pert', 'NPWLC')
SAVE_PDF = True
SAVE_CSV = True
DEBUG = False

Style.apply()
mpl.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'mathtext.fontset': 'cm',
    'legend.frameon': True,
    'legend.framealpha': 0.85,
    'axes.unicode_minus': False,
})

prim.MODEL_COLORS['Pert'] = 'tab:blue'
prim.MODEL_COLORS['NPWLC'] = 'tab:green'

Y_BINS = make_bins_from_width(-5.0, 5.0, 0.5)
PT_BINS = make_bins_from_width(0.0, 20.0, 2.5)
PT_WINDOW = (0.0, 15.0)
CENT_CLASSES = [(0,10),(10,20),(20,40),(40,60),(60,80),(80,100)]
OUTDIR = project / 'primordial_output'
OUTDIR.mkdir(exist_ok=True, parents=True)

In [3]:
glauber_root = project / 'primordial_output' / 'glauber_data' / '8TeV_eloss'
prim.generate_primordial_glauber_maps(
    glauber_root,
    cfg=prim.GlauberBridgeConfig(roots_gev=8160.0, target_a=208, system='pA', bmax_fm=20.0, nb=401, include_npart=True, verbose=False),
)
prim._validate_glauber_maps(glauber_root)
print('[ok] glauber maps ready:', glauber_root)

[ok] glauber maps ready: /home/sawin/Desktop/Charmonia/charmonia_combined_analysis/primordial_output/glauber_data/8TeV_eloss


In [4]:
cfg = ReaderConfig(debug=DEBUG)
summary = []
all_results = {}

for form in FORMS:
    print(f'\n===== FORM: {form} =====')
    combos = prim._load_primordial_combos(ENERGY, form, glauber_root, cfg)
    combos = [c for c in combos if c.model in MODELS]
    assert combos, f'No combos for form={form}'
    print('[ok] combos:', ', '.join([f'{c.model}:{c.form}' for c in combos]))
    for c in combos:
        maps = prim._maps_from_runs(c.runs)
        probe = maps.b_to_nbin(np.array([0.0, 2.0, 4.0], dtype=float))
        assert np.all(np.isfinite(probe)), f'Non-finite nbin for {form}/{c.model}'
        print(f'[ok] {form}/{c.model} nbin(b=0,2,4)={probe[0]:.3f},{probe[1]:.3f},{probe[2]:.3f}')

    prim_y = prim.primordial_vs_y_all(combos, prim.STATES, CENT_CLASSES, PT_WINDOW, Y_BINS)
    prim_pt = prim.primordial_vs_pT_all(combos, prim.STATES, CENT_CLASSES, prim.Y_WINDOWS, PT_BINS)
    prim_cent = prim.primordial_vs_cent_from_vs_y(prim_y, prim.Y_WINDOWS, prim.STATES, prim.MODELS)
    assert prim_y and prim_pt and prim_cent
    print('[ok] computed y / pT / centrality tables')

    tauform = prim._tauform_for_form(form)
    prim._save_vs_y_plots_and_csv(prim_y, OUTDIR, ENERGY, form, tauform, SAVE_PDF, SAVE_CSV)
    prim._save_vs_pt_plots_and_csv(prim_pt, OUTDIR, ENERGY, form, tauform, PT_WINDOW, SAVE_PDF, SAVE_CSV)
    prim._save_vs_cent_plots(prim_cent, OUTDIR, ENERGY, form, tauform, CENT_CLASSES, SAVE_PDF)
    print('[ok] outputs saved')

    all_results[form] = {'combos': combos, 'prim_y': prim_y, 'prim_pt': prim_pt, 'prim_cent': prim_cent}
    summary.append({'form': form, 'models_loaded': ','.join([c.model for c in combos]), 'n_models': len(combos)})

pd.DataFrame(summary)


===== FORM: new =====
[ok] combos: Pert:new, NPWLC:new
[ok] new/Pert nbin(b=0,2,4)=15.498,14.688,11.829
[ok] new/NPWLC nbin(b=0,2,4)=15.498,14.688,11.829
[ok] computed y / pT / centrality tables
[ok] outputs saved

===== FORM: old =====
[ok] combos: Pert:old, NPWLC:old
[ok] old/Pert nbin(b=0,2,4)=15.498,14.688,11.829
[ok] old/NPWLC nbin(b=0,2,4)=15.498,14.688,11.829
[ok] computed y / pT / centrality tables
[ok] outputs saved

===== FORM: radius =====
[ok] combos: Pert:radius, NPWLC:radius
[ok] radius/Pert nbin(b=0,2,4)=15.498,14.688,11.829
[ok] radius/NPWLC nbin(b=0,2,4)=15.498,14.688,11.829
[ok] computed y / pT / centrality tables
[ok] outputs saved


,form,models_loaded,n_models
0,new,"Pert,NPWLC",2
1,old,"Pert,NPWLC",2
2,radius,"Pert,NPWLC",2


In [5]:
tag = ENERGY.replace('.', 'p') + 'TeV'
pdfs = sorted(OUTDIR.glob(f'primordial_*_{tag}_*.pdf'))
csvs = sorted(OUTDIR.glob(f'primordial_*_{tag}_*.csv'))
print(f'PDF count: {len(pdfs)}')
print(f'CSV count: {len(csvs)}')
print('Sample PDFs:')
for p in pdfs[:6]:
    print(' -', p.name)
print('Sample CSVs:')
for p in csvs[:6]:
    print(' -', p.name)

PDF count: 45
CSV count: 36
Sample PDFs:
 - primordial_RpA_vs_cent_chicJ_1P_8p16TeV_new.pdf
 - primordial_RpA_vs_cent_chicJ_1P_8p16TeV_old.pdf
 - primordial_RpA_vs_cent_chicJ_1P_8p16TeV_radius.pdf
 - primordial_RpA_vs_cent_jpsi_1S_8p16TeV_new.pdf
 - primordial_RpA_vs_cent_jpsi_1S_8p16TeV_old.pdf
 - primordial_RpA_vs_cent_jpsi_1S_8p16TeV_radius.pdf
Sample CSVs:
 - primordial_RpA_vs_pT_chicJ_1P_backward_8p16TeV_new.csv
 - primordial_RpA_vs_pT_chicJ_1P_backward_8p16TeV_old.csv
 - primordial_RpA_vs_pT_chicJ_1P_backward_8p16TeV_radius.csv
 - primordial_RpA_vs_pT_chicJ_1P_central_8p16TeV_new.csv
 - primordial_RpA_vs_pT_chicJ_1P_central_8p16TeV_old.csv
 - primordial_RpA_vs_pT_chicJ_1P_central_8p16TeV_radius.csv
